# Hushwheel Fixture RAG Lab

This notebook turns the whimsical hushwheel corpus into a repeatable retrieval lab using the same tested helpers that back the repository tests.

## 1. Load The Shared Fixture Scaffold

### 1.1 Keep Fixture Logic In Tested Python Modules

The notebook should orchestrate the experiment, not duplicate benchmark, validation, or corpus-summary logic inline.

In [1]:
from pathlib import Path

from repo_rag_lab.notebook_scaffolding import build_hushwheel_fixture_lab_context
from repo_rag_lab.notebook_support import (
    assert_contains_text,
    assert_minimum_pass_rate,
    assert_no_validation_issues,
    resolve_repo_root,
    write_notebook_run_log,
)

root = resolve_repo_root(Path.cwd().resolve())
context = build_hushwheel_fixture_lab_context(root)
context["fixture_root"]


INFO repo_rag_lab.notebook_scaffolding Built hushwheel fixture context for 6 examples across 19 documents.


'tests/fixtures/hushwheel_lexiconarium'

## 2. Inspect Corpus Scale And Benchmark Inputs

### 2.1 Confirm The Fixture Is Large Enough To Stress Retrieval

Review the generated manifest, the corpus summary, and the question-suite summary before interpreting answers.

In [2]:
{
    "manifest": context["fixture_manifest"],
    "corpus_summary": context["corpus_summary"],
    "training_summary": context["training_summary"],
    "population_summary": context["population_summary"],
}


{'manifest': {'entry_count': 4108,
  'source_size_bytes': 1725649,
  'doc_files': ['README.md',
   'docs/concepts.md',
   'docs/operations.md',
   'docs/districts.md',
   'docs/catalog.md',
   'docs/architecture.md',
   'docs/testing.md',
   'docs/packaging.md'],
  'quality_targets': ['lint',
   'unit',
   'integration',
   'bdd',
   'check',
   'dist',
   'install',
   'uninstall'],
  'package_metadata': 'packaging/hushwheel.package.json'},
 'corpus_summary': {'document_count': 19,
  'chunk_count': 1650,
  'largest_source': 'hushwheel.c'},
 'training_summary': {'example_count': 6,
  'benchmark_count': 6,
  'questions': ['What is the ember index?',
   'What does the lantern vowel measure?',
   'How does print_prefix_matches handle prefix search?',
   'Why does the archive keep a moss ledger?',
   'What does the stats command report?',
   'Which file defines the GlossaryEntry struct?'],
  'unique_tags': ['code',
   'command',
   'concept',
   'header',
   'hushwheel',
   'lore',
   'pre

## 3. Compare Concept And Code Retrieval

### 3.1 Review One Lore Question And One Implementation Question

The shared scaffold highlights a concept-oriented question and a code-oriented question so ranking shifts stay easy to interpret.

In [3]:
[
    {
        "question": run["question"],
        "context_sources": run["context_sources"],
        "answer_preview": run["answer"][:280],
    }
    for run in context["highlight_runs"]
]


[{'question': 'What is the ember index?',
  'context_sources': ['README.md',
   'README.md',
   'docs/concepts.md',
   'src/hushwheel.c'],
  'answer_preview': 'Question: What is the ember index?\n\nEvidence:\n- /home/standard/dspy_rag_in_repo_docs_and_impl1/tests/fixtures/hushwheel_lexiconarium/README.md: | `docs/architecture.md` | Design notes for the data table, CLI dispatch, and test seam. | | `docs/testing.md` | Test strategy and make-'},
 {'question': 'How does print_prefix_matches handle prefix search?',
  'context_sources': ['src/hushwheel.c',
   'docs/architecture.md',
   'README.md',
   'docs/operations.md'],
  'answer_preview': 'Question: How does print_prefix_matches handle prefix search?\n\nEvidence:\n- /home/standard/dspy_rag_in_repo_docs_and_impl1/tests/fixtures/hushwheel_lexiconarium/src/hushwheel.c: "ember index: %d\\n", entry->ember_index); printf("lantern vowel: %d\\n", lantern_vowel_count(entry->term'}]

## 4. Assert Benchmark Health And Log The Run

### 4.1 Fail Fast On Retrieval Regressions

Keep the notebook useful as a playbook by asserting benchmark health, checking the highlighted answers, and writing a notebook log.

In [4]:
assert_no_validation_issues(
    context["training_validation_issues"],
    context="hushwheel training samples",
)
assert_no_validation_issues(
    context["population_validation_issues"],
    context="hushwheel population samples",
)
assert_minimum_pass_rate(context["benchmark_summary"], minimum_pass_rate=1.0)
assert_contains_text(
    context["highlight_runs"][0]["answer"],
    ["ember index"],
    context=context["highlight_runs"][0]["question"],
)
assert_contains_text(
    context["highlight_runs"][1]["answer"],
    ["print_prefix_matches", "prefix search"],
    context=context["highlight_runs"][1]["question"],
)
log_path = write_notebook_run_log(root, "05-hushwheel-fixture-rag-lab", context)
{
    "benchmark_summary": context["benchmark_summary"],
    "reranked_sources": context["reranked_sources"],
    "log_path": str(log_path.relative_to(root)),
}


{'benchmark_summary': {'case_count': 6,
  'pass_count': 6,
  'pass_rate': 1.0,
  'retrieved_source_hits': {'README.md': 5,
   'docs/architecture.md': 2,
   'docs/catalog.md': 3,
   'docs/concepts.md': 1,
   'docs/operations.md': 2,
   'docs/testing.md': 2,
   'include/hushwheel.h': 1,
   'src/hushwheel.c': 3},
  'matched_source_hits': {'README.md': 4,
   'docs/concepts.md': 1,
   'docs/operations.md': 2,
   'include/hushwheel.h': 1,
   'src/hushwheel.c': 3},
  'results': [{'question': 'What is the ember index?',
    'expected_sources': ['README.md', 'docs/concepts.md', 'src/hushwheel.c'],
    'retrieved_sources': ['README.md', 'docs/concepts.md', 'src/hushwheel.c'],
    'matched_sources': ['README.md', 'docs/concepts.md', 'src/hushwheel.c'],
    'passed': True,
    'tags': ['hushwheel', 'concept', 'ranking']},
   {'question': 'What does the lantern vowel measure?',
    'expected_sources': ['README.md', 'docs/concepts.md', 'src/hushwheel.c'],
    'retrieved_sources': ['README.md', 'docs